Lecture 8

In [6]:
# import packages
import os
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred
import time

In [2]:
# 1. Look for the .env file and load the hidden variables inside it
load_dotenv()

# 2. Securely fetch the key using os.getenv()
api_key = os.getenv('FRED_API_KEY')

# 3. Initialize the FRED API connection
fred = Fred(api_key=api_key)

Now that we have the keys to the database, let’s actually download some data!

FRED organizes its massive database using Series IDs. Every single economic indicator has a unique, short code. For example:

- Gross domestic product is GDP
- Consumer Price Index (Inflation) is CPIAUCSL
- The national unemployment rate is UNRATE

Let’s pull the historical data for the Unemployment Rate. The fredapi library makes this incredibly simple with a single command: get_series().

Run this code in your next cell:

In [3]:
# Fetch the Unemployment Rate data
unemployment_data = fred.get_series('UNRATE')

# Display the first 5 rows of the data
unemployment_data.head()

1948-01-01    3.4
1948-02-01    3.8
1948-03-01    4.0
1948-04-01    3.9
1948-05-01    3.5
dtype: float64

In [4]:
# When you run this code, Python reaches out to the FRED servers, grabs the monthly unemployment data, and downloads it directly into your computer’s memory.
# Note that the data is saved as a pandas.Series, with a DatetimeIndex:
print(type(unemployment_data))
print(type(unemployment_data.index)) 

<class 'pandas.Series'>
<class 'pandas.DatetimeIndex'>


In [ ]:
# multiple data set 

# In FRED, state unemployment rates follow a standard naming convention: the two-letter state abbreviation followed by UR (CAUR, TXUR, etc.).

# Instead of writing the same download command four times, we will put our IDs in a list, loop through them, and use the pandas library to concatenate (combine) them into a single DataFrame.

# 1. Define a list of the exact FRED Series IDs you want
series_ids = ['UNRATE', 'CAUR', 'TXUR', 'NYUR', 'FLUR']

# 2. Create an empty list to store our downloaded pandas Series
series_list = []

# 3. Loop through the list of IDs
for series_id in series_ids:
    print(f"Downloading data for {series_id}...")
    
    # Fetch the data
    s = fred.get_series(series_id)
    
    # Name the series so the final DataFrame knows what to call the column
    s.name = series_id
    
    # Append the processed series to our list
    series_list.append(s)

    # Pause for half a second before starting the next iteration
    time.sleep(0.5)

# 4. Concatenate all series in the list into a single DataFrame
df_unemployment = pd.concat(series_list, axis=1)
# The pd.concat() function is a core tool in the pandas library used to combine multiple Series or DataFrames together. The word “concat” is short for concatenation, which simply means linking items together in a sequence.


# 5. Display the most recent 5 rows using .tail()
print("\nData successfully combined!")
df_unemployment.tail()


# Stick to the default join='outer' when downloading. This ensures you keep all the data and don’t accidentally drop any data upfront. You can decide later on how to deal with missing values, depending on what you need for your analysis.


Data successfully combined!


,UNRATE,CAUR,TXUR,NYUR,FLUR
2025-11-01,4.5,5.5,4.3,4.6,4.3
2025-12-01,4.4,5.5,4.3,4.6,4.3
2026-01-01,4.3,5.4,4.3,4.6,4.5
2026-02-01,4.4,5.4,4.3,4.6,4.6
2026-03-01,4.3,NaN,NaN,NaN,NaN


To prevent this, you must explicitly slow down your code. You can achieve this using Python’s built-in time module. By adding a time.sleep() command inside your loop, you instruct Python to pause execution for a fraction of a second before making the next request, ensuring you stay safely under the server’s speed limit.

When you download a series using get_series(), you only get the dates and the numbers. But what if you forget the exact definition of the data? Is the GDP measured in millions or billions? Is the data seasonally adjusted?

To find this out, you can use the get_series_info() method. This method returns the official metadata from the FRED database (as a pandas Series).

In [8]:
# Fetch the metadata for the UNRATE series
unrate_info = fred.get_series_info('UNRATE')

# Display the information
print(unrate_info)

id                                                                      UNRATE
realtime_start                                                      2026-04-03
realtime_end                                                        2026-04-03
title                                                        Unemployment Rate
observation_start                                                   1948-01-01
observation_end                                                     2026-03-01
frequency                                                              Monthly
frequency_short                                                              M
units                                                                  Percent
units_short                                                                  %
seasonal_adjustment                                        Seasonally Adjusted
seasonal_adjustment_short                                                   SA
last_updated                                        

In [9]:
print(unrate_info['notes'])

The unemployment rate represents the number of unemployed as a percentage of the labor force. Labor force data are restricted to people 16 years of age and older, who currently reside in 1 of the 50 states or the District of Columbia, who do not reside in institutions (e.g., penal and mental facilities, homes for the aged), and who are not on active duty in the Armed Forces.  This rate is also defined as the U-3 measure of labor underutilization.  The series comes from the 'Current Population Survey (Household Survey)'  The source code is: LNS14000000


In [11]:
import pandas as pd

# 1. Define the list of exact FRED Series IDs we are working with
series_ids = ['UNRATE','CAUR', 'TXUR', 'NYUR', 'FLUR']

# 2. Create an empty list. We will use this to store a "row" of data for each state.
rows = []

# 3. Loop through each ID to fetch and extract its metadata
for series_id in series_ids:
    
    # Fetch the metadata (returns a pandas Series with all attributes)
    info = fred.get_series_info(series_id)              
    
    # Extract only the fields we want, format them as a dictionary, 
    # and append that dictionary to our 'rows' list.
    rows.append({
        "series_id":            info["id"],
        "title":                info["title"],
        "units":                info["units"],
        "frequency":            info["frequency"],
        "seasonal_adjustment":  info["seasonal_adjustment"],
        "observation_start":    info["observation_start"],
        "observation_end":      info["observation_end"]
    })

# 4. Convert our list of dictionaries directly into a pandas DataFrame
df_info = pd.DataFrame(rows)




df_info.head()

,series_id,title,units,frequency,seasonal_adjustment,observation_start,observation_end
0,UNRATE,Unemployment Rate,Percent,Monthly,Seasonally Adjusted,1948-01-01,2026-03-01
1,CAUR,Unemployment Rate in California,Percent,Monthly,Seasonally Adjusted,1976-01-01,2026-02-01
2,TXUR,Unemployment Rate in Texas,Percent,Monthly,Seasonally Adjusted,1976-01-01,2026-02-01
3,NYUR,Unemployment Rate in New York,Percent,Monthly,Seasonally Adjusted,1976-01-01,2026-02-01
4,FLUR,Unemployment Rate in Florida,Percent,Monthly,Seasonally Adjusted,1976-01-01,2026-02-01
